# Distributed Cluster Job Manager, Update Demo
Prototype showing job submission, scheduling, execution, and monitoring logic works.

In [2]:
from jobsubmission import JobSubmissionManager
from scheduler import ClusterManager, Scheduler
from executor import Executor
from monitor import Monitor
from persistence import Persistence
from interface import NotebookUI

# Initialize all modules
job_mgr = JobSubmissionManager()
cluster = ClusterManager()
cluster.create_local_cluster(num_workers=3, cores_per_node=4, ram_per_node=4096)

scheduler = Scheduler(cluster, job_mgr)
executor = Executor(cluster)
monitor = Monitor(scheduler, cluster, job_mgr)
persist = Persistence()

print("System ready.")

[CLUSTER] Created 3 local workers with 4 cores, 4096MB RAM each
System ready.


## Interactive Interface
Use the tabs below to submit jobs, upload YAML/JSON files, view status, and monitor the cluster.

In [3]:
ui = NotebookUI(job_mgr, cluster, scheduler, executor, monitor, persist)
ui.display_all()

[EXEC] Running SubTask(job=a2fdf62e, chunk=0/4, node=worker-0, status=running) on worker-0
[EXEC] Running SubTask(job=a2fdf62e, chunk=1/4, node=worker-1, status=running) on worker-1
[EXEC] Running SubTask(job=a2fdf62e, chunk=2/4, node=worker-2, status=running) on worker-2
[EXEC] Running SubTask(job=a2fdf62e, chunk=3/4, node=worker-0, status=running) on worker-0
[EXEC] SubTask(job=a2fdf62e, chunk=0/4, node=worker-0, status=completed) finished in 0.124s (code=0)
[EXEC] SubTask(job=a2fdf62e, chunk=1/4, node=worker-1, status=completed) finished in 0.124s (code=0)
[EXEC] SubTask(job=a2fdf62e, chunk=3/4, node=worker-0, status=completed) finished in 0.124s (code=0)
[EXEC] SubTask(job=a2fdf62e, chunk=2/4, node=worker-2, status=completed) finished in 0.124s (code=0)


## Test 1 — Simple single-node job

In [4]:
job1 = job_mgr.submit({
    "job_name": "hello_world",
    "command": "echo Hello from the cluster",
})

scheduler.schedule_job(job1)
active = [t for t in scheduler.active_tasks if t.parent_job_id == job1.job_id]
executor.run_job_subtasks(active)
for t in active:
    scheduler.release_resources(t)
scheduler.handle_job_completion(job1.job_id)

print(f"State: {job1.state}")
print(f"Output: {job1.result}")

[SUBMITTED] Job(75b0877f, hello_world, state=queued)
[SCHED] Quota for hello_world: {'max_cores': 6, 'max_ram_mb': 6144}
[SCHED] Decomposed hello_world into 1 sub-tasks
[SCHED] hello_world running on nodes: ['worker-0']
[EXEC] Running SubTask(job=75b0877f, chunk=0/1, node=worker-0, status=running) on worker-0
[EXEC] SubTask(job=75b0877f, chunk=0/1, node=worker-0, status=completed) finished in 0.018s (code=0)
[SCHED] Job hello_world completed successfully
State: completed
Output: ['Hello from the cluster --chunk-index 0 --total-chunks 1']


## Test 2 — Distributable job across 3 nodes

In [5]:
job2 = job_mgr.submit({
    "job_name": "distributed_task",
    "command": "echo Processing data chunk",
    "resources": {"cpus": 3, "memory_mb": 1536},
    "distributable": True,
    "chunks": 3,
})

scheduler.schedule_job(job2)
active = [t for t in scheduler.active_tasks if t.parent_job_id == job2.job_id]
executor.run_job_subtasks(active)
for t in active:
    scheduler.release_resources(t)
scheduler.handle_job_completion(job2.job_id)

print(f"State: {job2.state}")
print(f"Output: {job2.result}")
print(f"Nodes used: {job2.assigned_nodes}")

[SUBMITTED] Job(76aaecda, distributed_task, state=queued)
[SCHED] Quota for distributed_task: {'max_cores': 6, 'max_ram_mb': 6144}
[SCHED] Decomposed distributed_task into 3 sub-tasks
[SCHED] distributed_task running on nodes: ['worker-2', 'worker-1', 'worker-0']
[EXEC] Running SubTask(job=76aaecda, chunk=0/3, node=worker-0, status=running) on worker-0
[EXEC] Running SubTask(job=76aaecda, chunk=1/3, node=worker-1, status=running) on worker-1
[EXEC] Running SubTask(job=76aaecda, chunk=2/3, node=worker-2, status=running) on worker-2
[EXEC] SubTask(job=76aaecda, chunk=1/3, node=worker-1, status=completed) finished in 0.021s (code=0)
[EXEC] SubTask(job=76aaecda, chunk=2/3, node=worker-2, status=completed) finished in 0.021s (code=0)
[EXEC] SubTask(job=76aaecda, chunk=0/3, node=worker-0, status=completed) finished in 0.022s (code=0)
[SCHED] Job distributed_task completed successfully
State: completed
Output: ['Processing data chunk --chunk-index 0 --total-chunks 3', 'Processing data chunk -

## Test 3 — Invalid job (validation)

In [6]:
try:
    job_mgr.submit({"job_name": "", "command": ""})
except ValueError as e:
    print(f"Caught validation error:\n{e}")

Caught validation error:
Job validation failed:
  - Missing required field: 'job_name'
  - Missing required field: 'command'


## System Summary

In [7]:
summary = monitor.get_cluster_summary()
for k, v in summary.items():
    if k != "cluster":
        print(f"{k}: {v}")

print("\nCluster nodes:")
for name, info in summary["cluster"].items():
    print(f"  {name}: {info['utilization']}% utilized, {info['tasks']} active tasks")

total_jobs: 3
queued: 0
running: 0
completed: 3
failed: 0
active_tasks: 0
queued_tasks: 0

Cluster nodes:
  worker-0: 0.0% utilized, 0 active tasks
  worker-1: 0.0% utilized, 0 active tasks
  worker-2: 0.0% utilized, 0 active tasks


## Save State

In [8]:
persist.save_jobs(job_mgr)
persist.save_cluster_state(cluster.get_cluster_status())
print("State saved.")

[PERSIST] Saved 3 jobs
State saved.
